# Redeploy VM Notebook

Notebook nay dung khi GPU VM bi kill hoac can deploy lai Hunyuan worker. Chay tung cell tren Jupyter cua VM.

Runtime hien tai: Expo -> FastAPI backend -> local cleanup -> Hunyuan3D worker -> GLB. Khong dung Gemini/Nano Banana va khong dung TripoSR.

## 0. Reopen Jupyter after VM stop

When a Google Cloud VM is stopped, JupyterLab and every quick Cloudflare tunnel stop too. Start them again from SSH before running this notebook.

Open Google Cloud Console -> Compute Engine -> VM instances -> SSH, then run the commands below.


### 0A. Start JupyterLab in tmux

Run this in SSH tab 1:

```bash
tmux ls || true
tmux new -s jupyter
cd ~/work
source venv/bin/activate
source ~/.bashrc
jupyter lab --no-browser --ip=127.0.0.1 --port=8888 --ServerApp.allow_remote_access=True
```

If it says `duplicate session: jupyter`, attach to the existing session instead:

```bash
tmux attach -t jupyter
```

Copy the token printed by Jupyter. Detach without stopping Jupyter with `Ctrl+B`, then `D`.


### 0B. Expose Jupyter through Cloudflare

Run this in SSH tab 2:

```bash
cloudflared tunnel --url http://127.0.0.1:8888
```

Open this on Windows/Chrome:

```text
https://YOUR_JUPYTER_TUNNEL.trycloudflare.com/lab?token=YOUR_JUPYTER_TOKEN
```

Then open this notebook from the repo path:

```text
~/work/AI_3D_Reconstruction_Systerm/deploy/REDEPLOY_VM.ipynb
```


### 0C. Fast restart if workspace already exists

If the VM disk still has `~/work`, you usually only need to restart Jupyter/tunnels and then run sections 3-7 below. If `~/work` is missing or the VM is fresh, run section 2 to clone/bootstrap again.


## 1. Check GPU and OS

In [ ]:
!nvidia-smi
!python3 --version
!free -h
!df -h | head

## 2. Clone repo and bootstrap Hunyuan worker

Cell nay cai Python venv, Hunyuan3D-2, worker FastAPI, va tao systemd service `hunyuan-worker`.

In [ ]:
%%bash
set -euo pipefail
mkdir -p ~/work
cd ~/work
if [ ! -d AI_3D_Reconstruction_Systerm/.git ]; then
  git clone https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git
fi
cd AI_3D_Reconstruction_Systerm
git pull --ff-only
bash scripts/gcp_hunyuan_worker_bootstrap.sh

## 2B. Ensure worker service exists after VM restart

After a VM stop/start, processes are gone. If `hunyuan-worker.service` already exists, systemd should restart it. If the service does not exist, this cell re-runs the bootstrap script and creates it again. Run this cell before checking worker health.


In [ ]:
%%bash
set -euo pipefail

cd ~/work/AI_3D_Reconstruction_Systerm
git fetch origin
git checkout codex/hunyuan-shape-then-paint
git pull --ff-only

if systemctl list-unit-files | grep -q "^hunyuan-worker.service"; then
  echo "hunyuan-worker.service exists; restarting it."
  sudo systemctl restart hunyuan-worker
else
  echo "hunyuan-worker.service is missing; running bootstrap to create it."
  bash scripts/gcp_hunyuan_worker_bootstrap.sh
fi

sudo systemctl status hunyuan-worker --no-pager

echo "Waiting for worker health endpoint..."
for attempt in $(seq 1 60); do
  body=$(curl -fsS --max-time 5 http://127.0.0.1:8010/health 2>/tmp/hunyuan_health_curl.err || true)
  if [ -n "$body" ] && echo "$body" | python3 -m json.tool; then
    echo "Worker health OK after ${attempt}s."
    exit 0
  fi
  if ! sudo systemctl is-active --quiet hunyuan-worker; then
    echo "hunyuan-worker stopped while waiting for health. Recent logs:"
    sudo journalctl -u hunyuan-worker -n 80 --no-pager
    exit 1
  fi
  sleep 1
done

echo "Worker did not return valid JSON health within 60 seconds. Recent logs:"
sudo journalctl -u hunyuan-worker -n 120 --no-pager
exit 1


If this cell fails during package install or CUDA extension build, scroll to the first real error above. Usual causes are missing `nvcc`, broken NVIDIA driver, version drift, or Torch shared libraries not being visible to native extensions (`libc10.so`). The bootstrap script re-pins PyTorch CUDA cu126, pins `setuptools<82`, exports Torch `lib`, and preloads Torch shared libraries before testing texture extensions.


## 3. Check worker health and logs

In [ ]:
!curl -s http://127.0.0.1:8010/health

In [ ]:
!sudo systemctl status hunyuan-worker --no-pager

## 4. Start Cloudflare tunnel in tmux

Sau khi cell nay chay, xem log tunnel va copy URL `https://....trycloudflare.com` vao backend `.env.local`.

In [ ]:
%%bash
set -euo pipefail
cd ~/work/AI_3D_Reconstruction_Systerm
bash deploy/scripts/start_tunnel_tmux.sh
tmux capture-pane -t tunnel -p -S -80

## 5. View worker or tunnel logs live in Jupyter

In [ ]:
import subprocess, time
from IPython.display import clear_output

LOG_TARGET = "worker"  # use "worker" for systemd logs or "tunnel" for Cloudflare tmux logs

while True:
    clear_output(wait=True)
    if LOG_TARGET == "worker":
        cmd = ["sudo", "journalctl", "-u", "hunyuan-worker", "-n", "120", "--no-pager"]
    elif LOG_TARGET == "tunnel":
        cmd = ["tmux", "capture-pane", "-t", "tunnel", "-p", "-S", "-120"]
    else:
        raise ValueError('LOG_TARGET must be "worker" or "tunnel"')
    print(subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT))
    time.sleep(2)

## 6. Backend env on Windows

Copy tunnel URL vao `.env.local` cua backend Windows:

```env
RECONSTRUCTION_BACKEND=hunyuan_remote
HUNYUAN_REMOTE_URL=https://YOUR_TUNNEL.trycloudflare.com
HUNYUAN_REMOTE_OUTPUT_FORMAT=glb
HUNYUAN_REMOTE_ENABLE_TEXTURE=false
HUNYUAN_REMOTE_TIMEOUT_SECONDS=1800
HUNYUAN_REMOTE_POLL_INTERVAL_SECONDS=5
IMAGE_CLEANER_BACKEND=auto
ENABLE_REMBG_CLEANER=true
CLEAN_IMAGE_MAX_SIDE=1536
CLEAN_IMAGE_PAD_RATIO=0.08
```

Restart backend Windows:

```powershell
.\deploy\scripts\start_backend_windows.ps1 -HostIp 192.168.1.5
```

## 7. Smoke tests

Tren Windows backend:

```powershell
curl.exe http://127.0.0.1:8000/health
curl.exe https://YOUR_TUNNEL.trycloudflare.com/health
```

Sau do mo Expo va reconstruct lai tu mobile.